# 05 - Build SFT data from the fixed train split

This notebook rebuilds the supervised fine-tuning data from the **current** `train_set.csv`.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/cxr_eval_data"
p = os.path.join(BASE, "train_set.csv")
print(("OK      " if os.path.exists(p) else "MISSING ") + p)
if not os.path.exists(p):
    raise FileNotFoundError("Upload train_set.csv into the Drive folder cxr_eval_data, then rerun this cell.")


Mounted at /content/drive
OK      /content/drive/MyDrive/cxr_eval_data/train_set.csv


In [2]:
# Build SFT data from the current train split (train only).
import json, os
import pandas as pd, numpy as np

BASE = "/content/drive/MyDrive/cxr_eval_data"

PC = ['No Finding','Enlarged Cardiomediastinum','Cardiomegaly','Lung Lesion','Lung Opacity','Edema',
      'Consolidation','Pneumonia','Atelectasis','Pneumothorax','Pleural Effusion','Pleural Other',
      'Fracture','Support Devices']

DIRECT_PREFIX = """You are a radiologist assistant. For each of the 14 findings below, provide a single label: 1 (Present), 0 (Absent), -1 (Uncertain), or NaN (Not Mentioned).

You must output exactly 14 lines, one for each finding. Even if a finding is not mentioned in the report, you must still output its line with the value NaN. Do not skip any line.

Output format (one finding per line):
No Finding=value
Enlarged Cardiomediastinum=value
Cardiomegaly=value
Lung Lesion=value
Lung Opacity=value
Edema=value
Consolidation=value
Pneumonia=value
Atelectasis=value
Pneumothorax=value
Pleural Effusion=value
Pleural Other=value
Fracture=value
Support Devices=value

Report:
"""
DIRECT_SUFFIX = "\n\nOutput:"

STRUCT_PREFIX = """You are a radiologist assistant. Analyze the chest X-ray report and output JSON only. Use: present, absent, uncertain, or not_mentioned for each finding.

Template:
{"lungs_and_airways":{"lung_opacity":"","consolidation":"","edema":"","atelectasis":"","lung_lesion":"","pneumonia":""},"pleura":{"pleural_effusion":"","pneumothorax":"","pleural_other":""},"cardiovascular":{"cardiomegaly":"","enlarged_cardiomediastinum":""},"bones_and_chest_wall":{"fracture":""},"devices":{"support_devices":""},"general":{"no_finding":""}}

Report:
"""
STRUCT_SUFFIX = "\n\nJSON:"

def direct_completion(row):
    out = []
    for p in PC:
        v = row[p]
        if pd.isna(v): s = "NaN"
        elif v == 1.0: s = "1"
        elif v == 0.0: s = "0"
        elif v == -1.0: s = "-1"
        else: raise ValueError(f"unexpected value {v!r} for {p}")
        out.append(f"{p}={s}")
    assert len(out) == 14
    return "\n".join(out)

def struct_completion(row):
    def w(v):
        if pd.isna(v): return "not_mentioned"
        if v == 1.0: return "present"
        if v == 0.0: return "absent"
        if v == -1.0: return "uncertain"
        raise ValueError(f"unexpected value {v!r}")
    d = {
        "lungs_and_airways": {
            "lung_opacity": w(row["Lung Opacity"]), "consolidation": w(row["Consolidation"]),
            "edema": w(row["Edema"]), "atelectasis": w(row["Atelectasis"]),
            "lung_lesion": w(row["Lung Lesion"]), "pneumonia": w(row["Pneumonia"]),
        },
        "pleura": {
            "pleural_effusion": w(row["Pleural Effusion"]), "pneumothorax": w(row["Pneumothorax"]),
            "pleural_other": w(row["Pleural Other"]),
        },
        "cardiovascular": {
            "cardiomegaly": w(row["Cardiomegaly"]),
            "enlarged_cardiomediastinum": w(row["Enlarged Cardiomediastinum"]),
        },
        "bones_and_chest_wall": {"fracture": w(row["Fracture"])},
        "devices": {"support_devices": w(row["Support Devices"])},
        "general": {"no_finding": w(row["No Finding"])},
    }
    return json.dumps(d, separators=(",", ":"))

def parse_direct(text):
    parsed = {}
    for line in text.strip().split("\n"):
        line = line.strip()
        if "=" in line:
            parts = line.split("=")
            if len(parts) == 2:
                name, value = parts[0].strip(), parts[1].strip()
                if name in PC and value in ["1", "0", "-1", "NaN", "nan"]:
                    parsed[name] = value
    return [parsed.get(p, "NaN") for p in PC]

PATH_COLS = {
    "lungs_and_airways.lung_opacity": "Lung Opacity", "lungs_and_airways.consolidation": "Consolidation",
    "lungs_and_airways.edema": "Edema", "lungs_and_airways.atelectasis": "Atelectasis",
    "lungs_and_airways.lung_lesion": "Lung Lesion", "lungs_and_airways.pneumonia": "Pneumonia",
    "pleura.pleural_effusion": "Pleural Effusion", "pleura.pneumothorax": "Pneumothorax",
    "pleura.pleural_other": "Pleural Other", "cardiovascular.cardiomegaly": "Cardiomegaly",
    "cardiovascular.enlarged_cardiomediastinum": "Enlarged Cardiomediastinum",
    "bones_and_chest_wall.fracture": "Fracture", "devices.support_devices": "Support Devices",
    "general.no_finding": "No Finding",
}
VMAP = {"present": "1", "absent": "0", "uncertain": "-1", "not_mentioned": "NaN", "nan": "NaN"}

def parse_struct(text):
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e <= s: return ["NaN"] * 14
    try: data = json.loads(text[s:e+1])
    except Exception: return ["NaN"] * 14
    out = {}
    for path, p in PATH_COLS.items():
        cur = data
        for k in path.split("."):
            cur = cur.get(k) if isinstance(cur, dict) else None
        out[p] = VMAP.get(cur.strip().lower(), "NaN") if isinstance(cur, str) else "NaN"
    return [out.get(p, "NaN") for p in PC]

def canon(v):
    return np.nan if pd.isna(v) else float(v)

def canon_pred(s):
    return np.nan if s in ("NaN", "nan") else float(s)

train = pd.read_csv(os.path.join(BASE, "train_set.csv"))
assert len(train) == 2768, len(train)
assert train["id"].is_unique
missing = [p for p in PC if p not in train.columns]
assert not missing, missing

direct_rows, struct_rows = [], []
mismatches = []
for _, row in train.iterrows():
    report = str(row["full_report"])
    cd = direct_completion(row)
    cs = struct_completion(row)
    pd_ = parse_direct(cd)
    ps_ = parse_struct(cs)
    for i, p in enumerate(PC):
        t = canon(row[p])
        for pred, name in ((pd_[i], "direct"), (ps_[i], "structure")):
            v = canon_pred(pred)
            if not ((np.isnan(t) and np.isnan(v)) or (t == v)):
                mismatches.append((int(row["id"]), name, p, t, v))
    direct_rows.append({"id": int(row["id"]), "prompt": DIRECT_PREFIX + report + DIRECT_SUFFIX, "completion": cd})
    struct_rows.append({"id": int(row["id"]), "prompt": STRUCT_PREFIX + report + STRUCT_SUFFIX, "completion": cs})

if mismatches:
    for m in mismatches[:10]: print("MISMATCH", m)
    raise ValueError(f"{len(mismatches)} label round-trip mismatches - nothing was written")

for name, rows in (("sft_direct.jsonl", direct_rows), ("sft_structure.jsonl", struct_rows)):
    with open(os.path.join(BASE, name), "w", encoding="utf-8") as f:
        for x in rows:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")

with open(os.path.join(BASE, "prompt_templates.json"), "w", encoding="utf-8") as f:
    json.dump({
        "direct": {"prefix": DIRECT_PREFIX, "suffix": DIRECT_SUFFIX},
        "structure": {"prefix": STRUCT_PREFIX, "suffix": STRUCT_SUFFIX},
    }, f, ensure_ascii=False, indent=1)

print(f"Built direct={len(direct_rows)}  structure={len(struct_rows)}  mismatches=0")
print(f"Max prompt chars: direct={max(len(x['prompt']) for x in direct_rows)}  structure={max(len(x['prompt']) for x in struct_rows)}")
print(f"Saved to {BASE}: sft_direct.jsonl, sft_structure.jsonl, prompt_templates.json")

print("\n--- First direct example (prompt tail + completion) ---")
print(direct_rows[0]["prompt"][-180:])
print(direct_rows[0]["completion"])
print("\n--- First structure example (completion) ---")
print(struct_rows[0]["completion"])


Built direct=2768  structure=2768  mismatches=0
Max prompt chars: direct=1880  structure=1756
Saved to /content/drive/MyDrive/cxr_eval_data: sft_direct.jsonl, sft_structure.jsonl, prompt_templates.json

--- First direct example (prompt tail + completion) ---
le chest radiograph with no acute or progressive abnormality. There is no suggestion of enlargement of the known right retrohilar pulmonary nodule or pulmonary nodularity.

Output:
No Finding=NaN
Enlarged Cardiomediastinum=NaN
Cardiomegaly=1
Lung Lesion=0
Lung Opacity=0
Edema=NaN
Consolidation=NaN
Pneumonia=NaN
Atelectasis=NaN
Pneumothorax=NaN
Pleural Effusion=0
Pleural Other=NaN
Fracture=NaN
Support Devices=NaN

--- First structure example (completion) ---
{"lungs_and_airways":{"lung_opacity":"absent","consolidation":"not_mentioned","edema":"not_mentioned","atelectasis":"not_mentioned","lung_lesion":"absent","pneumonia":"not_mentioned"},"pleura":{"pleural_effusion":"absent","pneumothorax":"not_mentioned","pleural_other":"not_menti